# Extract Report Metadata
Extract metadata from Power BI reports including:
- Report pages
- Visuals (types, configurations, data bindings)
- Data sources and datasets
- Filters, slicers, and parameters
- Visual-level field usage

Uses **Fabric REST API** to get report definitions.

## 1. Imports

**Note**: Packages loaded from Fabric Environment (no installation needed).

In [11]:
# All packages available from attached Fabric Environment
import json
import base64
import requests
import sempy_labs.report
from datetime import datetime
from time import sleep
from typing import Dict, List, Any, Optional
from notebookutils import mssparkutils
from sempy.fabric import list_reports
from sempy_labs.report import connect_report


print("✅ Imports successful (from environment)")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 19, Finished, Available, Finished, False)

✅ Imports successful (from environment)


## 2. Configuration

In [12]:
# These will be provided as parameters when triggered via Livy
# For testing, set manually:
WORKSPACE_IDS = ["2b1d454b-61a1-4abb-96c3-2b476d945a96"]  # Can extract from multiple workspaces
LAKEHOUSE_ID = "8f644af3-54bc-4525-8cf8-7b78e5b01cd5"

EXTRACT_DETAILED_VISUALS = True

print(f"Extracting from {len(WORKSPACE_IDS)} workspace(s)")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 20, Finished, Available, Finished, False)

Extracting from 1 workspace(s)


## 3. Extract Report Metadata

In [13]:
def get_fabric_headers() -> dict:
    token = mssparkutils.credentials.getToken("pbi")
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }


def get_report_definition(workspace_id: str, report_id: str) -> dict:
    """Fetch the public report definition and handle Fabric LRO responses."""
    headers = get_fabric_headers()
    api_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/reports/{report_id}/getDefinition"
    response = requests.post(api_url, headers=headers)

    if response.status_code == 200:
        return response.json()

    if response.status_code != 202:
        response.raise_for_status()

    operation_url = response.headers.get("Location")
    retry_after = int(response.headers.get("Retry-After", "5"))

    if not operation_url:
        raise ValueError("Fabric getDefinition returned 202 without a Location header.")

    while True:
        sleep(retry_after)
        operation_response = requests.get(operation_url, headers=headers)
        operation_response.raise_for_status()
        operation_payload = operation_response.json()

        if "definition" in operation_payload:
            return operation_payload

        status = operation_payload.get("status")
        if status == "Succeeded":
            result_response = requests.get(f"{operation_url}/result", headers=headers)
            result_response.raise_for_status()
            return result_response.json()

        if status == "Failed":
            raise ValueError(f"Fabric getDefinition failed for report {report_id}: {operation_payload}")

        retry_after = int(operation_response.headers.get("Retry-After", response.headers.get("Retry-After", "5")))


def get_report_storage_format(workspace_id: str, report_id: str) -> str:
    """Return PBIR or PBIR-Legacy based on the report definition parts."""
    try:
        definition_response = get_report_definition(workspace_id, report_id)
        definition = definition_response.get("definition", {})
        definition_format = definition.get("format")

        if definition_format:
            return definition_format

        parts = definition.get("parts", [])
        part_paths = {part.get("path", "") for part in parts}

        if any(path.startswith("definition/") for path in part_paths):
            return "PBIR"
        if "report.json" in part_paths:
            return "PBIR-Legacy"
    except Exception as definition_error:
        print(f"  ⚠️  Could not determine report format: {definition_error}")

    return "Unknown"


def extract_report_with_sempy(report_id: str, workspace_id: str, extract_detailed: bool = True) -> dict:
    """
    Extract report metadata using sempy_labs.report.connect_report (PBIR format required).
    """
    extraction_start = datetime.now()

    with connect_report(report=report_id, workspace=workspace_id, readonly=True) as report:
        report_name = report.Name
        dataset_id = report.DatasetId if hasattr(report, "DatasetId") else None

        print(f"  ✅ sempy success - Report: {report_name}")
        print(f"  ℹ️  Dataset: {dataset_id}")

        pages_data = []
        visual_count = 0

        if extract_detailed:
            print("  🔍 Extracting pages and visuals...")

            try:
                pages_df = report.list_pages()

                if not pages_df.empty:
                    visuals_df = report.list_visuals()

                    for _, page_row in pages_df.iterrows():
                        page_name = page_row.get("Page Name", page_row.get("name", "Unknown"))
                        page_display_name = page_row.get("Display Name", page_row.get("displayName", page_name))

                        page_info = {
                            "name": page_name,
                            "displayName": page_display_name,
                            "ordinal": page_row.get("Ordinal", page_row.get("ordinal", 0)),
                            "width": page_row.get("Width", page_row.get("width")),
                            "height": page_row.get("Height", page_row.get("height")),
                            "hidden": page_row.get("Hidden", page_row.get("hidden", False)),
                            "visuals": [],
                            "visualCount": 0
                        }

                        if not visuals_df.empty:
                            if "Page Name" in visuals_df.columns:
                                page_visuals = visuals_df[visuals_df["Page Name"] == page_name]
                            elif "pageName" in visuals_df.columns:
                                page_visuals = visuals_df[visuals_df["pageName"] == page_name]
                            else:
                                page_visuals = visuals_df

                            for _, visual_row in page_visuals.iterrows():
                                visual_info = {
                                    "name": visual_row.get("Visual Name", visual_row.get("name")),
                                    "title": visual_row.get("Visual Title", visual_row.get("title")),
                                    "type": visual_row.get("Visual Type", visual_row.get("type")),
                                    "x": visual_row.get("X", visual_row.get("x")),
                                    "y": visual_row.get("Y", visual_row.get("y")),
                                    "width": visual_row.get("Width", visual_row.get("width")),
                                    "height": visual_row.get("Height", visual_row.get("height")),
                                    "z": visual_row.get("Z", visual_row.get("z"))
                                }
                                page_info["visuals"].append(visual_info)
                                visual_count += 1

                            page_info["visualCount"] = len(page_info["visuals"])
                            print(f"    └─ Page '{page_display_name}': {page_info['visualCount']} visual(s)")

                        pages_data.append(page_info)

            except Exception as detail_error:
                print(f"  ⚠️  Could not extract detailed pages/visuals: {str(detail_error)}")

        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()

        visuals_per_page = [
            {
                "pageName": page["displayName"],
                "visualCount": page["visualCount"]
            }
            for page in pages_data
        ] if pages_data else []

        detailed_report_info = {
            "id": report_id,
            "name": report_name,
            "displayName": report_name,
            "description": report.Description if hasattr(report, "Description") else None,
            "datasetId": dataset_id,
            "reportType": report.ReportType if hasattr(report, "ReportType") else None,
            "modifiedBy": report.ModifiedBy if hasattr(report, "ModifiedBy") else None,
            "modifiedDateTime": report.ModifiedDateTime if hasattr(report, "ModifiedDateTime") else None,
            "createdDateTime": report.CreatedDateTime if hasattr(report, "CreatedDateTime") else None,
            "webUrl": report.WebUrl if hasattr(report, "WebUrl") else None,
            "embedUrl": report.EmbedUrl if hasattr(report, "EmbedUrl") else None
        }

        return {
            "artifactId": report_id,
            "artifactType": "Report",
            "workspaceId": workspace_id,
            "timestamp": extraction_end.isoformat(),
            "data": {
                "reportInfo": detailed_report_info,
                "pages": pages_data,
                "counts": {
                    "pages": len(pages_data),
                    "visuals": visual_count,
                    "visualsPerPage": visuals_per_page
                }
            },
            "metadata": {
                "extractionDuration": duration,
                "extractionMethod": "sempy_labs.report.connect_report",
                "detailedExtraction": extract_detailed,
                "reportStorageFormat": "PBIR",
                "status": "success"
            }
        }


def extract_report_with_api(report_id: str, workspace_id: str) -> dict:
    """
    Extract comprehensive report metadata using Fabric REST API getDefinition endpoint.
    Parses report definition to extract pages, visuals, and filters (works with any report format).
    """
    extraction_start = datetime.now()
    headers = get_fabric_headers()

    # Get basic report info
    api_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/reports/{report_id}"
    response = requests.get(api_url, headers=headers)
    response.raise_for_status()
    report_data = response.json()
    
    report_name = report_data.get("displayName") or report_data.get("name", "Unknown")
    dataset_id = report_data.get("datasetId")

    print(f"  ✅ REST API success - Report: {report_name}")
    print(f"  ℹ️  Dataset: {dataset_id}")
    print(f"  🔍 Parsing report definition for pages and visuals...")

    # Get report definition with pages/visuals structure
    definition_response = get_report_definition(workspace_id, report_id)
    definition = definition_response.get("definition", {})
    parts = definition.get("parts", [])
    
    # Parse definition parts to extract pages, visuals, and filters
    pages_data = []
    visual_count = 0
    report_filters = []
    
    try:
        # Create a dictionary of decoded parts for easy access
        decoded_parts = {}
        for part in parts:
            path = part.get("path", "")
            payload = part.get("payload")
            if payload:
                try:
                    decoded_content = base64.b64decode(payload).decode('utf-8')
                    decoded_parts[path] = json.loads(decoded_content)
                except Exception as decode_error:
                    print(f"    ⚠️  Could not decode part '{path}': {decode_error}")
        
        # Extract report-level filters from report.json
        report_json = decoded_parts.get("report.json") or decoded_parts.get("definition/report.json")
        if report_json:
            report_filters = report_json.get("filters", [])
            if report_filters:
                print(f"    └─ Found {len(report_filters)} report-level filter(s)")
        
        # Determine format and extract pages
        is_pbir = any(path.startswith("definition/") for path in decoded_parts.keys())
        
        if is_pbir:
            # PBIR format: pages are in separate files under definition/pages/
            pages_metadata = decoded_parts.get("definition/pages.json", {})
            page_order = pages_metadata.get("order", [])
            
            for page_name in page_order:
                page_json_path = f"definition/pages/{page_name}/page.json"
                page_json = decoded_parts.get(page_json_path)
                
                if page_json:
                    page_info = {
                        "name": page_json.get("name", page_name),
                        "displayName": page_json.get("displayName", page_name),
                        "ordinal": len(pages_data),
                        "width": page_json.get("width"),
                        "height": page_json.get("height"),
                        "hidden": page_json.get("hidden", False),
                        "filters": page_json.get("filters", []),
                        "visuals": [],
                        "visualCount": 0
                    }
                    
                    # Extract visuals from this page
                    visual_folder_prefix = f"definition/pages/{page_name}/visuals/"
                    for part_path, part_content in decoded_parts.items():
                        if part_path.startswith(visual_folder_prefix) and part_path.endswith("/visual.json"):
                            visual_json = part_content
                            visual_name = visual_json.get("name", "Unknown")
                            
                            visual_info = {
                                "name": visual_name,
                                "title": visual_json.get("title"),
                                "type": visual_json.get("visualType"),
                                "x": visual_json.get("x"),
                                "y": visual_json.get("y"),
                                "width": visual_json.get("width"),
                                "height": visual_json.get("height"),
                                "z": visual_json.get("z"),
                                "filters": visual_json.get("filters", []),
                                "query": visual_json.get("query")
                            }
                            page_info["visuals"].append(visual_info)
                            visual_count += 1
                    
                    page_info["visualCount"] = len(page_info["visuals"])
                    pages_data.append(page_info)
                    print(f"    └─ Page '{page_info['displayName']}': {page_info['visualCount']} visual(s)")
        
        else:
            # PBIR-Legacy format: all structure in report.json
            legacy_report = decoded_parts.get("report.json")
            if legacy_report:
                sections = legacy_report.get("sections", [])
                
                for idx, section in enumerate(sections):
                    page_name = section.get("name", f"Page{idx+1}")
                    page_display_name = section.get("displayName", page_name)
                    
                    page_info = {
                        "name": page_name,
                        "displayName": page_display_name,
                        "ordinal": idx,
                        "width": section.get("width"),
                        "height": section.get("height"),
                        "hidden": section.get("hidden", False),
                        "filters": section.get("filters", []),
                        "visuals": [],
                        "visualCount": 0
                    }
                    
                    # Extract visuals from this section
                    visual_containers = section.get("visualContainers", [])
                    for container in visual_containers:
                        config = container.get("config")
                        if config:
                            try:
                                config_json = json.loads(config) if isinstance(config, str) else config
                                visual_info = {
                                    "name": config_json.get("name", "Unknown"),
                                    "title": config_json.get("singleVisual", {}).get("visualType"),
                                    "type": config_json.get("singleVisual", {}).get("visualType"),
                                    "x": container.get("x"),
                                    "y": container.get("y"),
                                    "width": container.get("width"),
                                    "height": container.get("height"),
                                    "z": container.get("z"),
                                    "filters": config_json.get("filters", []),
                                    "query": config_json.get("query")
                                }
                                page_info["visuals"].append(visual_info)
                                visual_count += 1
                            except Exception as visual_error:
                                print(f"    ⚠️  Could not parse visual config: {visual_error}")
                    
                    page_info["visualCount"] = len(page_info["visuals"])
                    pages_data.append(page_info)
                    print(f"    └─ Page '{page_display_name}': {page_info['visualCount']} visual(s)")
    
    except Exception as parse_error:
        print(f"  ⚠️  Error parsing report definition: {parse_error}")
        import traceback
        print(f"  📋 Traceback: {traceback.format_exc()}")

    extraction_end = datetime.now()
    duration = (extraction_end - extraction_start).total_seconds()

    visuals_per_page = [
        {
            "pageName": page["displayName"],
            "visualCount": page["visualCount"]
        }
        for page in pages_data
    ]

    detailed_report_info = {
        "id": report_data.get("id", report_id),
        "name": report_name,
        "displayName": report_data.get("displayName", report_name),
        "description": report_data.get("description"),
        "datasetId": dataset_id,
        "reportType": report_data.get("reportType"),
        "modifiedBy": report_data.get("modifiedBy"),
        "modifiedDateTime": report_data.get("modifiedDateTime"),
        "createdDateTime": report_data.get("createdDateTime"),
        "webUrl": report_data.get("webUrl"),
        "embedUrl": report_data.get("embedUrl"),
        "subscriptions": report_data.get("subscriptions"),
        "users": report_data.get("users"),
        "datasetWorkspaceId": report_data.get("datasetWorkspaceId"),
        "appId": report_data.get("appId"),
        "isOwnedByMe": report_data.get("isOwnedByMe"),
        "isReadOnly": report_data.get("isReadOnly"),
        "isOriginalPbixReport": report_data.get("isOriginalPbixReport")
    }

    return {
        "artifactId": report_id,
        "artifactType": "Report",
        "workspaceId": workspace_id,
        "timestamp": extraction_end.isoformat(),
        "data": {
            "reportInfo": detailed_report_info,
            "pages": pages_data,
            "reportFilters": report_filters,
            "counts": {
                "pages": len(pages_data),
                "visuals": visual_count,
                "visualsPerPage": visuals_per_page,
                "reportFilters": len(report_filters)
            }
        },
        "metadata": {
            "extractionDuration": duration,
            "extractionMethod": "fabric_rest_api_with_definition_parsing",
            "detailedExtraction": True,
            "status": "success"
        }
    }


def extract_report_metadata(report_id: str, workspace_id: str, extract_detailed: bool = True) -> dict:
    """Route extraction explicitly based on the report storage format."""
    print(f"\n📊 Extracting Report: {report_id}")

    report_storage_format = get_report_storage_format(workspace_id, report_id)
    print(f"  ℹ️  Report storage format: {report_storage_format}")

    if report_storage_format == "PBIR":
        print("  🔄 Using sempy_labs.report.connect_report for PBIR report...")
        return extract_report_with_sempy(report_id, workspace_id, extract_detailed)

    print("  🔄 Using Fabric REST API for non-PBIR report...")
    result = extract_report_with_api(report_id, workspace_id)
    result["metadata"]["reportStorageFormat"] = report_storage_format
    return result


print("✅ Extraction functions defined:")
print("   • get_fabric_headers() - authorization headers")
print("   • get_report_definition() - fetch report definition parts with LRO handling")
print("   • get_report_storage_format() - detect PBIR vs PBIR-Legacy format")
print("   • extract_report_with_sempy() - PBIR format extraction using sempy_labs")
print("   • extract_report_with_api() - comprehensive extraction via REST API (parses pages/visuals/filters)")
print("   • extract_report_metadata() - main orchestrator with format-based routing")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 11, Finished, Available, Finished, False)

✅ Extraction functions defined:
   • get_fabric_headers() - authorization headers
   • get_report_definition() - fetch report definition parts with LRO handling
   • get_report_storage_format() - detect PBIR vs PBIR-Legacy format
   • extract_report_with_sempy() - PBIR format extraction using sempy_labs
   • extract_report_with_api() - comprehensive extraction via REST API (parses pages/visuals/filters)
   • extract_report_metadata() - main orchestrator with format-based routing


StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 21, Finished, Available, Finished, False)

✅ Extraction functions defined:
   • get_fabric_headers() - authorization headers
   • get_report_definition() - fetch report definition parts with LRO handling
   • get_report_storage_format() - detect PBIR vs PBIR-Legacy format
   • extract_report_with_sempy() - PBIR format extraction using sempy_labs
   • extract_report_with_api() - comprehensive extraction via REST API (parses pages/visuals/filters)
   • extract_report_metadata() - main orchestrator with format-based routing


## 4. Process All Workspaces

In [14]:
# Initialize results tracking
all_results = []

# Process each workspace
for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing workspace: {workspace_id}")
    
    try:
        # List all reports in workspace using sempy
        reports_df = list_reports(workspace=workspace_id)
        print(f"  ✅ Found {len(reports_df)} report(s)")
        print(f"  📋 Columns: {list(reports_df.columns)}")
        
        # Process each report
        for idx, report_row in reports_df.iterrows():
            report_id = report_row.get('Id') or report_row.get('id')
            report_name = report_row.get('Name') or report_row.get('DisplayName', 'Unknown')
            
            if not report_id:
                print(f"  ⚠️  Skipping report '{report_name}' - no valid ID")
                continue
            
            # Extract report metadata
            result = extract_report_metadata(
                report_id=report_id,
                workspace_id=workspace_id,
                extract_detailed=EXTRACT_DETAILED_VISUALS
            )
            all_results.append(result)
            
            # Save individual result to lakehouse (use relative paths)
            file_path = f"Files/lineage/raw/reports/{report_id}.json"
            
            # Ensure directory exists (relative path from workspace root)
            mssparkutils.fs.mkdirs("Files/lineage/raw/reports")
            
            # Write JSON
            json_str = json.dumps(result, indent=2)
            mssparkutils.fs.put(file_path, json_str, overwrite=True)
            print(f"    └─ Saved: {file_path}")
            
    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")
        import traceback
        print(f"  📋 Traceback: {traceback.format_exc()}")

print(f"\n✅ Extraction complete: {len(all_results)} report(s) processed")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 22, Finished, Available, Finished, False)


🔍 Processing workspace: 2b1d454b-61a1-4abb-96c3-2b476d945a96
  ✅ Found 2 report(s)
  📋 Columns: ['Id', 'Report Type', 'Format', 'Name', 'Web Url', 'Embed Url', 'Is From Pbix', 'Is Owned By Me', 'Dataset Id', 'App Id', 'Description', 'Dataset Workspace Id', 'Users', 'Subscriptions', 'Original Report Id']

📊 Extracting Report: a8247d4d-8590-4bd6-8521-0f84bbb73a77
  ℹ️  Report storage format: PBIR-Legacy
  🔄 Using Fabric REST API for non-PBIR report...
  ✅ REST API success - Report: Training Datamodel Solution
  ℹ️  Dataset: None
  🔍 Parsing report definition for pages and visuals...
    ⚠️  Could not decode part 'StaticResources/RegisteredResources/AdventureWorksLogo9335593362734207.jpg': 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
    └─ Page '1. Einfache Aggregation und Filter Context': 7 visual(s)
    └─ Saved: Files/lineage/raw/reports/a8247d4d-8590-4bd6-8521-0f84bbb73a77.json

📊 Extracting Report: 88e4eb9f-f94f-4618-9225-20921b8f614c
  ℹ️  Report storage 

## 5. Summary

In [15]:
success_count = sum(1 for r in all_results if r["metadata"]["status"] == "success")
success_limited_count = sum(1 for r in all_results if r["metadata"]["status"] == "success_limited")
error_count = len(all_results) - success_count - success_limited_count

print("\n" + "="*50)
print("EXTRACTION SUMMARY")
print("="*50)
print(f"Total Reports: {len(all_results)}")
print(f"✅ Full Success: {success_count} (with pages/visuals)")
print(f"⚠️  Limited Success: {success_limited_count} (basic metadata only)")
print(f"❌ Errors: {error_count}")
print("\nResults saved to: Files/lineage/raw/reports/")
print("="*50)

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 23, Finished, Available, Finished, False)


EXTRACTION SUMMARY
Total Reports: 2
✅ Full Success: 2 (with pages/visuals)
⚠️  Limited Success: 0 (basic metadata only)
❌ Errors: 0

Results saved to: Files/lineage/raw/reports/


---
# 🧪 Testing & Validation

## Test Single Report Before Full Extraction
Run the test cell below to verify extraction works correctly with one report before processing all reports in a workspace.

## 🧪 Test Single Report Extraction

In [16]:
# TEST: Extract a single report to verify extraction works correctly
# Update these IDs to test with your own report

TEST_WORKSPACE_ID = "2b1d454b-61a1-4abb-96c3-2b476d945a96"
TEST_REPORT_ID = "a8247d4d-8590-4bd6-8521-0f84bbb73a77"

if not TEST_WORKSPACE_ID or not TEST_REPORT_ID:
    print("⚠️  Set TEST_WORKSPACE_ID and TEST_REPORT_ID before running this test.")
    print("\nHow to get IDs:")
    print("1. Workspace ID: From workspace URL (last GUID)")
    print("   Example: https://app.fabric.microsoft.com/groups/{workspace-id}/list")
    print("2. Report ID: Open Report → Settings → About → Copy Report ID")
else:
    print("🔎 Testing report extraction...\n")
    print(f"Workspace ID: {TEST_WORKSPACE_ID}")
    print(f"Report ID: {TEST_REPORT_ID}\n")

    # Detect format and route
    report_storage_format = get_report_storage_format(TEST_WORKSPACE_ID, TEST_REPORT_ID)
    expected_method = "sempy_labs" if report_storage_format == "PBIR" else "REST API with parsing"
    
    print(f"Storage Format: {report_storage_format}")
    print(f"Expected Method: {expected_method}\n")

    # Extract report
    test_result = extract_report_metadata(
        report_id=TEST_REPORT_ID,
        workspace_id=TEST_WORKSPACE_ID,
        extract_detailed=EXTRACT_DETAILED_VISUALS
    )

    # Display results
    metadata = test_result.get("metadata", {})
    data = test_result.get("data", {})
    counts = data.get("counts", {})
    report_info = data.get("reportInfo", {})

    print("\n" + "="*60)
    print("📋 EXTRACTION RESULT")
    print("="*60)
    print(f"Status: {metadata.get('status', 'N/A')}")
    print(f"Method: {metadata.get('extractionMethod', 'N/A')}")
    print(f"Format: {metadata.get('reportStorageFormat', report_storage_format)}")
    print(f"Duration: {metadata.get('extractionDuration', 0):.2f}s")
    print(f"\nReport: {report_info.get('name', 'N/A')}")
    print(f"Dataset: {report_info.get('datasetId', 'N/A')}")
    print(f"Pages: {counts.get('pages', 0)}")
    print(f"Visuals: {counts.get('visuals', 0)}")
    
    if counts.get("reportFilters"):
        print(f"Report Filters: {counts.get('reportFilters', 0)}")

    if counts.get("visualsPerPage"):
        print("\n📊 Visual Distribution:")
        for page_info in counts["visualsPerPage"]:
            print(f"   • {page_info['pageName']}: {page_info['visualCount']} visual(s)")

    # Save result
    test_path = f"Files/lineage/test/test_{TEST_REPORT_ID}.json"
    mssparkutils.fs.mkdirs("Files/lineage/test")
    mssparkutils.fs.put(test_path, json.dumps(test_result, indent=2), overwrite=True)
    print(f"\n💾 Result saved to: {test_path}")
    print("="*60)

StatementMeta(, c49f1664-df42-4872-8c4f-85d6696b2159, 8, Finished, Available, Finished, False)

⚠️  Set TEST_WORKSPACE_ID and TEST_REPORT_ID before running this verification cell.


StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 24, Finished, Available, Finished, False)

🔎 Testing report extraction...

Workspace ID: 2b1d454b-61a1-4abb-96c3-2b476d945a96
Report ID: a8247d4d-8590-4bd6-8521-0f84bbb73a77

Storage Format: PBIR-Legacy
Expected Method: REST API with parsing


📊 Extracting Report: a8247d4d-8590-4bd6-8521-0f84bbb73a77
  ℹ️  Report storage format: PBIR-Legacy
  🔄 Using Fabric REST API for non-PBIR report...
  ✅ REST API success - Report: Training Datamodel Solution
  ℹ️  Dataset: None
  🔍 Parsing report definition for pages and visuals...
    ⚠️  Could not decode part 'StaticResources/RegisteredResources/AdventureWorksLogo9335593362734207.jpg': 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
    └─ Page '1. Einfache Aggregation und Filter Context': 7 visual(s)

📋 EXTRACTION RESULT
Status: success
Method: fabric_rest_api_with_definition_parsing
Format: PBIR-Legacy
Duration: 20.83s

Report: Training Datamodel Solution
Dataset: None
Pages: 1
Visuals: 7

📊 Visual Distribution:
   • 1. Einfache Aggregation und Filter Context: 7 

## 🧪 Verify Saved Data

In [17]:
# List and inspect all extracted report files
try:
    files = mssparkutils.fs.ls("/lakehouse/default/Files/lineage/raw/reports")
    print(f"📁 Found {len(files)} extracted report file(s):\n")
    
    for file in files:
        if file.name.endswith('.json'):
            content = mssparkutils.fs.head(f"/lakehouse/default/Files/lineage/raw/reports/{file.name}", 10000)
            data = json.loads(content)
            
            report_info = data.get('data', {}).get('reportInfo', {})
            metadata = data.get('metadata', {})
            counts = data.get('data', {}).get('counts', {})
            
            print(f"📄 {file.name}")
            print(f"   Name: {report_info.get('name', 'N/A')}")
            print(f"   Status: {metadata.get('status', 'N/A')}")
            print(f"   Method: {metadata.get('extractionMethod', 'N/A')}")
            print(f"   Content: {counts.get('pages', 0)} pages, {counts.get('visuals', 0)} visuals")
            
            if counts.get('visualsPerPage'):
                print(f"   Distribution: ", end="")
                print(", ".join([f"{p['pageName']}({p['visualCount']})" for p in counts['visualsPerPage'][:3]]))
            print()
            
except Exception as e:
    print(f"❌ No files found or error reading: {e}")
    print("   Run extraction first (cell 7)")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 25, Finished, Available, Finished, False)

❌ No files found or error reading: path: /lakehouse/default/Files/lineage/raw/reports is invalid, if you want to operate on the local path, please add the prefix file:, for example file:/lakehouse/default/Files/lineage/raw/reports. Exception: Operation failed: "Bad Request", 400, GET, http://onelake.dfs.fabric.microsoft.com/a559a09d-159b-43f9-a5f7-f908bc5a77bb?upn=false&resource=filesystem&maxResults=5000&directory=lakehouse/default/Files/lineage/raw/reports&timeout=90&recursive=false, FriendlyNameSupportDisabled, "Request Failed with WorkspaceId and ArtifactId should be either valid Guids or valid Names"
   Run extraction first (cell 7)


## 🧪 Cross-Reference with Semantic Models

In [ ]:
# Analyze which reports use which semantic models
try:
    report_files = mssparkutils.fs.ls("/lakehouse/default/Files/lineage/raw/reports")

    spark_session = globals().get("spark")
    if spark_session is None:
        from pyspark.sql import SparkSession
        spark_session = SparkSession.builder.getOrCreate()

    def read_report_json(full_path: str) -> dict:
        # Read full file content via Spark text reader to avoid truncation from fs.head().
        text_rows = spark_session.read.text(full_path).collect()
        full_text = "\n".join([row["value"] for row in text_rows])
        return json.loads(full_text)

    dataset_usage = {}

    for file in report_files:
        if file.name.endswith('.json'):
            full_path = f"/lakehouse/default/Files/lineage/raw/reports/{file.name}"
            try:
                data = read_report_json(full_path)
            except Exception as read_error:
                print(f"⚠️  Skipping {file.name}: {read_error}")
                continue

            if data.get('metadata', {}).get('status') == 'success':
                report_name = data['data']['reportInfo']['name']
                dataset_id = data['data']['reportInfo'].get('datasetId')

                if dataset_id:
                    if dataset_id not in dataset_usage:
                        dataset_usage[dataset_id] = []
                    dataset_usage[dataset_id].append(report_name)

    if dataset_usage:
        print("📊 Dataset → Report Linkage:\n")
        for dataset_id, reports in dataset_usage.items():
            print(f"Dataset {dataset_id[:8]}... ({len(reports)} report(s)):")
            for report in reports:
                print(f"   └─ {report}")
            print()
    else:
        print("No dataset linkages found in extracted reports")

except Exception as e:
    print(f"❌ Error analyzing linkages: {e}")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 26, Finished, Available, Finished, False)

❌ Error analyzing linkages: path: /lakehouse/default/Files/lineage/raw/reports is invalid, if you want to operate on the local path, please add the prefix file:, for example file:/lakehouse/default/Files/lineage/raw/reports. Exception: Operation failed: "Bad Request", 400, GET, http://onelake.dfs.fabric.microsoft.com/a559a09d-159b-43f9-a5f7-f908bc5a77bb?upn=false&resource=filesystem&maxResults=5000&directory=lakehouse/default/Files/lineage/raw/reports&timeout=90&recursive=false, FriendlyNameSupportDisabled, "Request Failed with WorkspaceId and ArtifactId should be either valid Guids or valid Names"


---
## 📋 Troubleshooting

### Common Issues:

**1. "403 Forbidden" or "401 Unauthorized"**
- Verify workspace permissions (Contributor or higher)
- Check report permissions (viewer minimum)
- Ensure correct workspace/report IDs

**2. "Report definition not accessible"**
- Both extraction methods now provide comprehensive metadata
- PBIR format: Best performance with sempy_labs (direct object access)
- PBIR-Legacy or Unknown: API extraction parses definition structure
- If API extraction fails, check report permissions and definition access

**3. "Empty pages/visuals arrays"**
- Check extraction status and error messages
- Verify getDefinition endpoint is accessible (requires appropriate permissions)
- Review parsed definition structure for unexpected format
- If parsing fails, check logs for specific decode/parse errors

**4. "Dataset ID is null"**
- Report might not be bound to a dataset (dashboards, etc.)
- This is expected for some report types

### Extraction Methods:

**Method 1: sempy_labs.report.connect_report (Preferred for PBIR)**
- Requires: Report in PBIR format
- Provides: Direct object access via Python API
- Status: "success"
- Best for: PBIR reports with native library support

**Method 2: Fabric REST API with Definition Parsing (Universal)**
- Requires: getDefinition endpoint access (any report format)
- Provides: Comprehensive metadata by parsing report definition (pages, visuals, filters, visual queries)
- Status: "success"
- Best for: PBIR-Legacy reports, Unknown formats, or when sempy connection fails
- Parses both PBIR (definition/ folder structure) and PBIR-Legacy (report.json) formats

### Expected Results:
- **Basic info**: Always captured via either method (name, description, dataset ID, timestamps)
- **Pages & Visuals**: Extracted by both methods (different approaches)
  - sempy: Direct DataFrame access
  - API: JSON parsing of definition structure
- **Filters**: Extracted at report, page, and visual levels
- **Visual Queries**: Available in definition structure (API method)
- **Performance**: sempy generally faster for PBIR, API method more universal

### Data Quality Notes:
- Visual field bindings available in visual query definitions
- Filter structures include field references and conditions
- Both methods provide equivalent visual counts and page structures
- API method includes additional metadata (positioning, z-index, visual config)

### Next Steps:
1. ✅ Test with one report (cell 11)
2. Verify lakehouse files (cell 13)
3. Check dataset linkages (cell 14)
4. Run full extraction if test succeeds (cell 7)
5. Compare extraction results for consistency
